# Pipeline Financeiro — inform_27 + Danone
Carrega dados de transporte, associa ingressos Danone, enriquece com coordenadas e quilometragem, calcula métricas de rentabilidade e exporta para Parquet (Power BI).

## 1. Configuração e ligação à base de dados

In [2]:
import platform
import sqlite3
import warnings

import pandas as pd

warnings.filterwarnings("ignore")


def get_paths() -> dict:
    """Devolve os caminhos de ficheiros consoante o sistema operativo."""
    sistema = platform.system()

    if sistema == "Windows":
        return {
            "db": r"C:\Users\LISARR\Documents\python\00.DB\2026.db",
            "parquet": r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet",
            "caminho_excel": r"C:\Users\LISARR\Documents\python\01.Financeiro\Danone_Custos_V3.xlsx",
        }

    if sistema == "Darwin":
        icloud = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        return {
            "db": f"{icloud}/00_DB/2026.db",
            "parquet": f"{icloud}/inform_27_2026_final.parquet",
            "caminho_excel": f"{icloud}/Danone_Custos_v3.xlsx",
        }

    return {
        "db": "2026.db",
        "parquet": "inform_27_2026_final.parquet",
        "excel": "Danone_Custos_v3.xlsx",
    }


PATHS = get_paths()

with sqlite3.connect(PATHS["db"]) as con:
    df = pd.read_sql_query("SELECT * FROM inform_27_2026", con)

print(f"Linhas carregadas: {len(df):,}")
df.head()

df_excel = pd.read_excel(PATHS["caminho_excel"])
print(f"Linhas Excel: {len(df_excel):,}")
df_excel.head()

Linhas carregadas: 797,698
Linhas Excel: 44,095


,BLIINF,CODACT,PREFPE,PFEENT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,CODMOP,...,Mes,Año,Ruta WH,Ruta Tte. Nueva,Unnamed: 26,Unnamed: 27,Unnamed: 28,Tarifa,Ingreso,Combustible
0,FGE50PTG,11,5021758091 413963878,20260102,350150908.0,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021758091 413963878,31.96,1.307803,1.402227
1,FGE50PTG,11,5021765321 413964675,20260102,350151650.0,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021765321 413964675,31.96,1.186355,1.272010
2,FGE50PTG,11,5021758092 413969049,20260102,350392489.0,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021758092 413969049,31.96,0.714626,0.766222
3,FGE50PTG,11,5021742353 413966257,20260102,350194778.0,PREÇO BAIXO - ANGEIRAS,1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021742353 413966257,31.96,0.884653,0.948525
4,FGE50PTG,11,5021756292 413963116,20260102,350390901.0,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,STD,MAS,...,1,2026,Porto Prevenda,Porto Prevenda,NaN,Porto Prevenda,5021756292 413963116,31.96,3.466382,3.716654


In [3]:
# ==========================
# 1. Converter para numérico
# ==========================
df["CODACT"] = pd.to_numeric(df["CODACT"], errors="coerce")
df["INGRESODT"] = pd.to_numeric(df["INGRESODT"], errors="coerce")
df["COSTEDT"] = pd.to_numeric(df["COSTEDT"], errors="coerce")
df["PALETS"] = pd.to_numeric(df["PALETS"], errors="coerce")

# ==========================
# 2. Validar tipos
# ==========================
df[["CODACT", "INGRESODT", "COSTEDT", "PALETS"]].dtypes

CODACT       float64
INGRESODT    float64
COSTEDT      float64
PALETS       float64
dtype: object

In [4]:
# ==========================
# 1. Total antes
# ==========================
total_antes = pd.to_numeric(df["INGRESODT"], errors="coerce").fillna(0).sum()

# ==========================
# 2. Zerar CODACT 11
# ==========================
df.loc[
    pd.to_numeric(df["CODACT"], errors="coerce").eq(11),
    "INGRESODT"
] = 0

# ==========================
# 3. Total depois
# ==========================
total_depois = pd.to_numeric(df["INGRESODT"], errors="coerce").fillna(0).sum()
diferenca = total_antes - total_depois

resultado = pd.DataFrame({
    "Métrica": [
        "INGRESODT antes",
        "INGRESODT depois",
        "INGRESODT removido"
    ],
    "Valor": [
        total_antes,
        total_depois,
        diferenca
    ]
})

resultado

,Métrica,Valor
0,INGRESODT antes,30086820.68
1,INGRESODT depois,29478890.63
2,INGRESODT removido,607930.05


In [5]:
# Clientes de Espanha

# ==========================
# 1. Ajustar ingresso clientes
# ==========================
clientes_ajuste = [338, 211, 15]

mask_clientes = pd.to_numeric(
    df["CODACT"],
    errors="coerce"
).isin(clientes_ajuste)

df.loc[mask_clientes, "INGRESODT"] = (
    pd.to_numeric(df.loc[mask_clientes, "COSTEDT"], errors="coerce") * 1.05
)

In [6]:
# ==========================
# 1. Ratear CODACT 13
# ==========================
'''mask_codact13 = df["CODACT"].eq(13)

n_linhas_codact13 = mask_codact13.sum()

valor_por_linha = 90_000 / n_linhas_codact13

df.loc[mask_codact13, "INGRESODT"] = valor_por_linha

# ==========================
# 2. Validar
# ==========================
pd.DataFrame({
    "metrica": [
        "Linhas CODACT 13",
        "Valor por linha",
        "Total INGRESODT CODACT 13",
    ],
    "valor": [
        n_linhas_codact13,
        valor_por_linha,
        df.loc[mask_codact13, "INGRESODT"].sum(),
    ],
})'''

'mask_codact13 = df["CODACT"].eq(13)\n\nn_linhas_codact13 = mask_codact13.sum()\n\nvalor_por_linha = 90_000 / n_linhas_codact13\n\ndf.loc[mask_codact13, "INGRESODT"] = valor_por_linha\n\n# ==========================\n# 2. Validar\n# ==========================\npd.DataFrame({\n    "metrica": [\n        "Linhas CODACT 13",\n        "Valor por linha",\n        "Total INGRESODT CODACT 13",\n    ],\n    "valor": [\n        n_linhas_codact13,\n        valor_por_linha,\n        df.loc[mask_codact13, "INGRESODT"].sum(),\n    ],\n})'

In [7]:
# ==========================
# 1. Identificar CODACT 13
# ==========================
codact_num = pd.to_numeric(df["CODACT"], errors="coerce")
mask_codact13 = codact_num.eq(13)

n_antes = len(df)
n_codact13 = mask_codact13.sum()

# ==========================
# 2. Excluir CODACT 13
# ==========================
df = df.loc[~mask_codact13].copy()

# ==========================
# 3. Validar
# ==========================
pd.DataFrame({
    "metrica": [
        "Linhas antes",
        "Linhas CODACT 13 excluídas",
        "Linhas depois",
        "CODACT 13 restantes",
    ],
    "valor": [
        n_antes,
        n_codact13,
        len(df),
        pd.to_numeric(df["CODACT"], errors="coerce").eq(13).sum(),
    ],
})

,metrica,valor
0,Linhas antes,797698
1,Linhas CODACT 13 excluídas,25566
2,Linhas depois,772132
3,CODACT 13 restantes,0


## 2. Dados base — `inform_27_2026`

In [8]:
# Conversão de tipos numéricos
colunas_numericas = [
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT",
    "PESO_BRUTO", "PALETS", "KM", "KMREALES",
]
for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["CODEUT"] = df["CODEUT"].astype(str)

# Remover espaços em branco de todas as colunas de texto
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Filtrar GESTION == "LIS", ou GESTION nulo com PROPIETARIO em CEP/PTG
df = df[
    (df["GESTION"] == "LIS")
    | (df["GESTION"].isna() & df["PROPIETARIO"].isin(["CEP", "PTG"]))
]

# Datas (após o trim, sem componente de hora)
df["FCARGA"] = pd.to_datetime(df["FCARGA"], format="%Y%m%d", errors="coerce").dt.date
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], format="%Y%m%d", errors="coerce").dt.date

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
94,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,230.939,NO,SAL_DAT027 (1).xls
95,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,TAM,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (1).xls
96,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,204.542,NO,SAL_DAT027 (1).xls
97,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,171.983,NO,SAL_DAT027 (1).xls
99,100,CEP,SALVESEN LOGISTICA AZAMBUJA 2 / SALVESEN LOGIS...,"TRANSPARENTODISSEIA - Transportes Unipessoal, ...",0000XXX,0000XXX,0.0,0.53,-0.53,0.02,...,328547,DL TRANSPARENTODISSEIA - Transportes Unipessoa...,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,311.828,NO,SAL_DAT027 (1).xls


## 3. Pipeline Danone — cálculo do ingresso por entrega

In [9]:
# ==========================
# 1. Ler KILOSPO
# ==========================
caminho_excel = PATHS["caminho_excel"]

raw_kilospo = pd.read_excel(caminho_excel, sheet_name="KILOSPO", header=None)
df_danone = raw_kilospo.iloc[1:].copy()
df_danone.columns = raw_kilospo.iloc[0]

# ==========================
# 2. Normalizar campos
# ==========================
df_danone["PREFPE"] = df_danone["PREFPE"].astype("string").str.strip()

if pd.api.types.is_datetime64_any_dtype(df_danone["PFEENT"]):
    df_danone["PFEENT"] = df_danone["PFEENT"].dt.strftime("%Y%m%d")
else:
    df_danone["PFEENT"] = df_danone["PFEENT"].astype("string").str.strip()

df_danone["PCODCL"] = df_danone["PCODCL"].astype("string").str.strip()
df_danone["tipo_local"] = df_danone["Ruta WH"].astype("string").str.strip()

df_danone["Ruta Tte. Nueva"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

df_danone["Kg Neto Entregado"] = pd.to_numeric(
    df_danone["Kg Neto Entregado"],
    errors="coerce"
)

# ==========================
# 3. Validar
# ==========================
print(f"Linhas carregadas da KILOSPO: {len(df_danone):,}")

df_danone[
    [
        "PREFPE",
        "PFEENT",
        "PCODCL",
        "PNOMCL",
        "tipo_local",
        "Ruta Tte. Nueva",
        "Kg Neto Entregado",
    ]
].head()


Linhas carregadas da KILOSPO: 44,095


,PREFPE,PFEENT,PCODCL,PNOMCL,tipo_local,Ruta Tte. Nueva,Kg Neto Entregado
1,5021758091 413963878,20260102,350150908,PREÇO BAIXO - DOIS AMIGOS,Porto Prevenda,Porto Prevenda,40.92
2,5021765321 413964675,20260102,350151650,PREÇO BAIXO SUP. AV. FER. AROS,Porto Prevenda,Porto Prevenda,37.12
3,5021758092 413969049,20260102,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",Porto Prevenda,Porto Prevenda,22.36
4,5021742353 413966257,20260102,350194778,PREÇO BAIXO - ANGEIRAS,Porto Prevenda,Porto Prevenda,27.68
5,5021756292 413963116,20260102,350390901,FROIZ - BRAGA II- RETAIL CENTE,Porto Prevenda,Porto Prevenda,108.46


In [10]:
# ==========================
# 1. Ler tarifas da BD
# ==========================
with sqlite3.connect(PATHS["db"]) as con:
    df_tarifas = pd.read_sql_query(
        """
        SELECT
            tipo_local,
            modelo_ingresso,
            tarifa_base
        FROM Danone_Tarifas_2026
        WHERE data_fim IS NULL
        """,
        con,
    )

# ==========================
# 2. Normalizar campos
# ==========================
df_tarifas["tipo_local"] = df_tarifas["tipo_local"].astype("string").str.strip()
df_tarifas["modelo_ingresso"] = df_tarifas["modelo_ingresso"].astype("string").str.strip()
df_tarifas["tarifa_base"] = pd.to_numeric(df_tarifas["tarifa_base"], errors="coerce")

df_danone["tipo_local"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

# ==========================
# 3. Aplicar tarifa
# ==========================
df_danone = df_danone.drop(
    columns=["modelo_ingresso", "tarifa_base"],
    errors="ignore"
)

df_danone = df_danone.merge(
    df_tarifas,
    on="tipo_local",
    how="left",
    validate="many_to_one",
)

# ==========================
# 4. Validar
# ==========================
print(f"Linhas: {len(df_danone):,}")
print(f"Com tarifa: {df_danone['tarifa_base'].notna().sum():,}")
print(f"Sem tarifa: {df_danone['tarifa_base'].isna().sum():,}")

df_danone[
    ["PREFPE", "Ruta Tte. Nueva", "tipo_local", "modelo_ingresso", "tarifa_base"]
].head(20)


Linhas: 44,095
Com tarifa: 30,793
Sem tarifa: 13,302


,PREFPE,Ruta Tte. Nueva,tipo_local,modelo_ingresso,tarifa_base
0,5021758091 413963878,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
1,5021765321 413964675,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
2,5021758092 413969049,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
3,5021742353 413966257,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
4,5021756292 413963116,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
5,5021762387 413967065,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
6,5021726862 413962594,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
7,5021766864 413973924,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
8,5021767276 413970112,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000
9,5021741336 413972570,Porto Prevenda,Porto Prevenda,POR_TONELADA,31.96000


In [11]:
# ==========================
# 1. Preparar cálculo
# ==========================
df_danone["Kg Neto Entregado"] = pd.to_numeric(
    df_danone["Kg Neto Entregado"],
    errors="coerce"
)

df_danone["ingresso_danone"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="Float64"
)

por_tonelada = df_danone["modelo_ingresso"].eq("POR_TONELADA")
por_entrega = df_danone["modelo_ingresso"].eq("POR_ENTREGA")
sem_ingresso = df_danone["modelo_ingresso"].eq("SEM_INGRESSO")

# ==========================
# 2. Calcular ingresso Danone
# ==========================
df_danone.loc[por_tonelada, "ingresso_danone"] = (
    df_danone.loc[por_tonelada, "Kg Neto Entregado"] / 1000
) * df_danone.loc[por_tonelada, "tarifa_base"]

df_danone.loc[por_entrega, "ingresso_danone"] = (
    df_danone.loc[por_entrega, "tarifa_base"]
)

df_danone.loc[sem_ingresso, "ingresso_danone"] = 0.0

# ==========================
# 3. Validar cálculo
# ==========================
df_danone[
    [
        "PREFPE",
        "tipo_local",
        "modelo_ingresso",
        "Kg Neto Entregado",
        "tarifa_base",
        "ingresso_danone",
    ]
].head(20)


,PREFPE,tipo_local,modelo_ingresso,Kg Neto Entregado,tarifa_base,ingresso_danone
0,5021758091 413963878,Porto Prevenda,POR_TONELADA,40.920,31.96000,1.307803
1,5021765321 413964675,Porto Prevenda,POR_TONELADA,37.120,31.96000,1.186355
2,5021758092 413969049,Porto Prevenda,POR_TONELADA,22.360,31.96000,0.714626
3,5021742353 413966257,Porto Prevenda,POR_TONELADA,27.680,31.96000,0.884653
4,5021756292 413963116,Porto Prevenda,POR_TONELADA,108.460,31.96000,3.466382
5,5021762387 413967065,Porto Prevenda,POR_TONELADA,113.750,31.96000,3.63545
6,5021726862 413962594,Porto Prevenda,POR_TONELADA,29.000,31.96000,0.92684
7,5021766864 413973924,Porto Prevenda,POR_TONELADA,143.040,31.96000,4.571558
8,5021767276 413970112,Porto Prevenda,POR_TONELADA,32.660,31.96000,1.043814
9,5021741336 413972570,Porto Prevenda,POR_TONELADA,31.900,31.96000,1.019524


In [12]:
# ==========================
# 1. Ingresso por mês
# ==========================
df_danone["PFEENT"] = pd.to_datetime(
    df_danone["PFEENT"],
    errors="coerce"
)

df_danone["mes"] = df_danone["PFEENT"].dt.month

soma_por_mes = (
    df_danone
    .groupby("mes", as_index=False)
    .agg(
        linhas=("ingresso_danone", "size"),
        ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1))
    )
)

soma_por_mes


,mes,linhas,ingresso_total
0,1,5522,122884.131421
1,2,5387,112978.531191
2,3,5730,133113.817669
3,4,5309,135126.357042
4,5,5293,133374.048236
5,6,5405,145315.388264
6,7,6072,168843.085034
7,8,5159,147566.346052
8,9,218,3999.145015


In [13]:
# ==========================
# 1. Tipo de dados CODACT
# ==========================
df["CODACT"].dtype

dtype('float64')

In [14]:
import calendar

mes8 = df_danone[df_danone["mes"] == 8]

dias_com_dados = mes8["PFEENT"].dt.day.nunique()
dias_total_mes = calendar.monthrange(2026, 8)[1]
total_mes8 = mes8["ingresso_danone"].sum(min_count=1)
media_diaria = total_mes8 / dias_com_dados
projecao_mes8 = media_diaria * dias_total_mes

print(f"Dias com dados em agosto: {dias_com_dados}")
print(f"Dias totais no mês: {dias_total_mes}")
print(f"Total registado até agora: {total_mes8:,.0f}")
print(f"Média diária: {media_diaria:,.0f}")
print(f"Projeção para o mês completo: {projecao_mes8:,.0f}")

Dias com dados em agosto: 26
Dias totais no mês: 31
Total registado até agora: 147,566
Média diária: 5,676
Projeção para o mês completo: 175,944


## 4. Associar o ingresso Danone ao dataframe principal

In [15]:
# ==========================
# 1. Preparar ingresso Danone
# ==========================
df_ingresso_danone = df_danone[["PREFPE", "PFEENT", "ingresso_danone"]].copy()

df_ingresso_danone["referencia_danone_chave"] = (
    df_ingresso_danone["PREFPE"]
    .astype("string")
    .str.extract(r"(\d{10})", expand=False)
)

df_ingresso_danone["PFEENT"] = pd.to_datetime(
    df_ingresso_danone["PFEENT"],
    errors="coerce"
)

df_ingresso_danone = df_ingresso_danone[
    df_ingresso_danone["referencia_danone_chave"].notna()
].copy()

# ==========================
# 2. Agregar por referência e data
# ==========================
df_ingresso_danone = (
    df_ingresso_danone
    .groupby(["referencia_danone_chave", "PFEENT"], dropna=False)
    .agg(ingresso_danone_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)

# ==========================
# 3. Preparar chaves do df principal
# ==========================
df["REFERENCIA"] = df["REFERENCIA"].astype("string").str.strip()
df["referencia_danone_chave"] = (
    df["REFERENCIA"]
    .str.extract(r"(\d{10})", expand=False)
)
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], errors="coerce")
df["_ordem_original"] = range(len(df))

# ==========================
# 4. Associar ingresso
# ==========================
df = df.merge(
    df_ingresso_danone,
    left_on=["referencia_danone_chave", "FENTREGA"],
    right_on=["referencia_danone_chave", "PFEENT"],
    how="left",
    validate="many_to_one",
).sort_values("_ordem_original").reset_index(drop=True)

# ==========================
# 5. Evitar duplicação do ingresso
# ==========================
tem_match = df["ingresso_danone_total"].notna()
ultima_linha = tem_match & ~df.duplicated(
    subset=["referencia_danone_chave", "FENTREGA"],
    keep="last"
)

df["ingresso_danone"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
df.loc[ultima_linha, "ingresso_danone"] = df.loc[ultima_linha, "ingresso_danone_total"]

df = df.drop(columns=["PFEENT", "ingresso_danone_total", "_ordem_original"])

# ==========================
# 6. Calcular ingresso total
# ==========================
df["total_ingresso"] = df["INGRESODT"].fillna(0) + df["ingresso_danone"].fillna(0)

# ==========================
# 7. Validar associação
# ==========================
total_calculado = df_danone["ingresso_danone"].sum(min_count=1)
total_associado = df["ingresso_danone"].sum(min_count=1)
diferenca = total_calculado - total_associado

print(f"Ingresso Danone calculado: {total_calculado:,.2f}")
print(f"Ingresso Danone associado: {total_associado:,.2f}")
print(f"Diferença: {diferenca:,.2f}")


Ingresso Danone calculado: 1,103,200.85
Ingresso Danone associado: 1,053,992.86
Diferença: 49,207.99


In [16]:
# ==========================
# 1. Mês de entrega
# ==========================
df["mes_entrega"] = df["FENTREGA"].dt.month

soma_por_mes_df = (
    df
    .groupby("mes_entrega")
    .agg(ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)

soma_por_mes_df["ingresso_total"] = (
    soma_por_mes_df["ingresso_total"]
    .round(0)
    .map("{:,.0f}".format)
)

soma_por_mes_df


,mes_entrega,ingresso_total
0,1,"118,027"
1,2,"108,390"
2,3,"128,004"
3,4,"129,383"
4,5,"128,055"
5,6,"138,234"
6,7,"159,736"
7,8,"140,459"
8,9,"3,705"


In [17]:
# ==========================
# 1. Resumo por atividade e mês de entrega
# ==========================
resumo_por_atividade_mes = (
    df[df["CODACT"].isin([11])]
    .groupby(["CODACT", "mes_entrega"])
    .agg(
        soma_ingresodt=("INGRESODT", "sum"),
        soma_ingresso_danone=("ingresso_danone", lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)

# ==========================
# 2. Formatar valores
# ==========================
resumo_por_atividade_mes["soma_ingresodt"] = (
    resumo_por_atividade_mes["soma_ingresodt"]
    .round(0)
    .map("{:,.0f}".format)
)

resumo_por_atividade_mes["soma_ingresso_danone"] = (
    resumo_por_atividade_mes["soma_ingresso_danone"]
    .round(0)
    .map("{:,.0f}".format)
)

resumo_por_atividade_mes


,CODACT,mes_entrega,soma_ingresodt,soma_ingresso_danone
0,11.0,1,0,"118,027"
1,11.0,2,0,"108,390"
2,11.0,3,0,"128,004"
3,11.0,4,0,"129,383"
4,11.0,5,0,"128,055"
5,11.0,6,0,"138,234"
6,11.0,7,0,"159,736"
7,11.0,8,0,"140,459"
8,11.0,9,0,"3,705"


In [18]:
# ==========================
# 1. Resumo por atividade e mês de entrega
# ==========================
resumo_por_atividade_mes = (
    df[df["CODACT"].isin([13])]
    .groupby(["CODACT", "mes_entrega"])
    .agg(
        soma_ingresodt=("INGRESODT", "sum"),
        soma_ingresso_danone=("ingresso_danone", lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)

# ==========================
# 2. Formatar valores
# ==========================
resumo_por_atividade_mes["soma_ingresodt"] = (
    resumo_por_atividade_mes["soma_ingresodt"]
    .round(0)
    .map("{:,.0f}".format)
)

resumo_por_atividade_mes["soma_ingresso_danone"] = (
    resumo_por_atividade_mes["soma_ingresso_danone"]
    .round(0)
    .map("{:,.0f}".format)
)

resumo_por_atividade_mes


,CODACT,mes_entrega,soma_ingresodt,soma_ingresso_danone


In [19]:
# ==========================
# 1. Resumo por atividade e mês de entrega
# ==========================
resumo_por_atividade_mes = (
    df[df["CODACT"].isin([311])]
    .groupby(["CODACT", "mes_entrega"])
    .agg(
        soma_ingresodt=("INGRESODT", "sum"),
        soma_ingresso_danone=("ingresso_danone", lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)

# ==========================
# 2. Formatar valores
# ==========================
resumo_por_atividade_mes["soma_ingresodt"] = (
    resumo_por_atividade_mes["soma_ingresodt"]
    .round(0)
    .map("{:,.0f}".format)
)

resumo_por_atividade_mes["soma_ingresso_danone"] = (
    resumo_por_atividade_mes["soma_ingresso_danone"]
    .round(0)
    .map("{:,.0f}".format)
)

resumo_por_atividade_mes


,CODACT,mes_entrega,soma_ingresodt,soma_ingresso_danone
0,311.0,1,"20,457",nan
1,311.0,2,"18,582",nan
2,311.0,3,"19,874",nan
3,311.0,4,"22,680",nan
4,311.0,5,"20,551",nan
5,311.0,6,"18,754",nan
6,311.0,7,"21,532",nan
7,311.0,8,"22,320",nan
8,311.0,9,"2,409",nan


## 5. Features derivadas

In [20]:
# Data, semana e dia da semana
df["data"] = pd.to_datetime(df["FCARGA"])
df["week_number"] = df["data"].dt.isocalendar().week
df["week_day"] = df["data"].dt.day_name()
df["mes"] = df["data"].dt.month
df["mes_nome"] = df["data"].dt.month_name()

df[["FCARGA", "week_number", "week_day", "mes_nome", "total_ingresso"]].head()

,FCARGA,week_number,week_day,mes_nome,total_ingresso
0,2026-02-01,5,Sunday,February,31.332933
1,2026-02-01,5,Sunday,February,13.3
2,2026-02-01,5,Sunday,February,105.402351
3,2026-02-01,5,Sunday,February,42.456427
4,2026-01-30,5,Friday,January,0.0


In [21]:
# Normalização da capacidade do camião (agrupar capacidades equivalentes)
mapeamento_capacidade = {
    4: 6, 5: 6, 6: 6,
    8: 12, 12: 12,
    14: 20, 15: 20, 16: 20, 18: 20, 20: 20,
    22: 24, 24: 24,
    33: 33, 66: 66,
}

df["CAMION_CAPACIDAD_NUM"] = pd.to_numeric(df["CAMION_CAPACIDAD"], errors="coerce")
df["capacidade_norm"] = df["CAMION_CAPACIDAD_NUM"].map(mapeamento_capacidade)

# Capacidade em falta (valor 0): preencher com a capacidade mais comum da mesma rota
linhas_sem_capacidade = df["CAMION_CAPACIDAD_NUM"] == 0
if linhas_sem_capacidade.any():
    capacidade_por_rota = (
        df.loc[df["CAMION_CAPACIDAD_NUM"] > 0]
        .groupby("CODEUT")["capacidade_norm"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.max())
    )
    for rota in df.loc[linhas_sem_capacidade, "CODEUT"].unique():
        if rota in capacidade_por_rota.index:
            df.loc[(df["CODEUT"] == rota) & linhas_sem_capacidade, "capacidade_norm"] = capacidade_por_rota[rota]

df["dados_validos"] = df["capacidade_norm"].notna()
print(f"Linhas com capacidade válida: {df['dados_validos'].sum():,} / {len(df):,}")

Linhas com capacidade válida: 95,652 / 101,271


## 6. Coordenadas geográficas (origem e destino)

In [22]:
# ==========================
# 1. Ler coordenadas
# ==========================
with sqlite3.connect(PATHS["db"]) as con:
    coords = pd.read_sql_query(
        """
        SELECT
            cp AS CP,
            point_x AS POINT_X,
            point_y AS POINT_Y
        FROM coordenadas
        """,
        con,
    )

# ==========================
# 2. Converter coordenadas
# ==========================
coords["POINT_X"] = pd.to_numeric(
    coords["POINT_X"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

coords["POINT_Y"] = pd.to_numeric(
    coords["POINT_Y"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# ==========================
# 3. Normalizar códigos postais
# ==========================
# ==========================
# 1. Normalizar códigos postais
# ==========================
def normalizar_cp(serie):
    cp = (
        serie
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str[:7]
    )

    invalido = (
        cp.str.len().fillna(0).lt(4)
        | cp.eq("0").fillna(False)
        | cp.eq("").fillna(False)
    )

    return cp.mask(invalido, pd.NA)

coords["CP_chave"] = normalizar_cp(coords["CP"])
coords["CP_parte"] = coords["CP_chave"].str[:4]

df["CPOSTAL_chave"] = normalizar_cp(df["CPOSTAL"])
df["CPOSTAD_chave"] = normalizar_cp(df["CPOSTAD"])

df["CPOSTAL_parte"] = df["CPOSTAL_chave"].str[:4]
df["CPOSTAD_parte"] = df["CPOSTAD_chave"].str[:4]

# ==========================
# 2. Validar
# ==========================
pd.DataFrame({
    "campo": ["coords", "CPOSTAL", "CPOSTAD"],
    "válidos": [
        coords["CP_chave"].notna().sum(),
        df["CPOSTAL_chave"].notna().sum(),
        df["CPOSTAD_chave"].notna().sum(),
    ],
    "inválidos": [
        coords["CP_chave"].isna().sum(),
        df["CPOSTAL_chave"].isna().sum(),
        df["CPOSTAD_chave"].isna().sum(),
    ],
})

# ==========================
# 4. Limpar colunas anteriores
# ==========================
df = df.drop(
    columns=[
        "longitude_origem",
        "latitude_origem",
        "longitude_destino",
        "latitude_destino",
        "longitude_origem_c",
        "latitude_origem_c",
        "longitude_destino_c",
        "latitude_destino_c",
    ],
    errors="ignore",
)

# ==========================
# 5. Coordenadas por CP exacto
# ==========================
coords_cp = (
    coords
    .dropna(subset=["CP_chave", "POINT_X", "POINT_Y"])
    .groupby("CP_chave", as_index=False)[["POINT_X", "POINT_Y"]]
    .mean()
)

coords_origem = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAL_chave",
        "POINT_X": "longitude_origem",
        "POINT_Y": "latitude_origem",
    }
)

coords_destino = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAD_chave",
        "POINT_X": "longitude_destino",
        "POINT_Y": "latitude_destino",
    }
)

df = df.merge(
    coords_origem,
    on="CPOSTAL_chave",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    coords_destino,
    on="CPOSTAD_chave",
    how="left",
    validate="many_to_one",
)

match_exacto_origem = df["longitude_origem"].notna().sum()
match_exacto_destino = df["longitude_destino"].notna().sum()

# ==========================
# 6. Fallback prefixo 4 dígitos
# ==========================
centroid = (
    coords
    .dropna(subset=["CP_parte", "POINT_X", "POINT_Y"])
    .groupby("CP_parte", as_index=False)[["POINT_X", "POINT_Y"]]
    .mean()
    .rename(
        columns={
            "POINT_X": "longitude_centroid",
            "POINT_Y": "latitude_centroid",
        }
    )
)

centroid_origem = centroid.rename(
    columns={
        "CP_parte": "CPOSTAL_parte",
        "longitude_centroid": "longitude_origem_c",
        "latitude_centroid": "latitude_origem_c",
    }
)

centroid_destino = centroid.rename(
    columns={
        "CP_parte": "CPOSTAD_parte",
        "longitude_centroid": "longitude_destino_c",
        "latitude_centroid": "latitude_destino_c",
    }
)

df = df.merge(
    centroid_origem,
    on="CPOSTAL_parte",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    centroid_destino,
    on="CPOSTAD_parte",
    how="left",
    validate="many_to_one",
)

df["longitude_origem"] = df["longitude_origem"].fillna(df["longitude_origem_c"])
df["latitude_origem"] = df["latitude_origem"].fillna(df["latitude_origem_c"])

df["longitude_destino"] = df["longitude_destino"].fillna(df["longitude_destino_c"])
df["latitude_destino"] = df["latitude_destino"].fillna(df["latitude_destino_c"])

# ==========================
# 7. Validar
# ==========================
print(f"Coordenadas válidas na BD: {coords[['POINT_X', 'POINT_Y']].dropna().shape[0]:,}")
print(f"Match exacto origem: {match_exacto_origem:,}")
print(f"Match exacto destino: {match_exacto_destino:,}")
print(f"Match final origem: {df['longitude_origem'].notna().sum():,}")
print(f"Match final destino: {df['longitude_destino'].notna().sum():,}")

Coordenadas válidas na BD: 178,409
Match exacto origem: 99,062
Match exacto destino: 77,275
Match final origem: 100,378
Match final destino: 94,146


## 7. Métricas de rentabilidade

In [23]:
df["custo_por_palete"] = df["COSTEDT"] / df["PALETS"].replace(0, 1)
df["ingresso_por_palete"] = df["total_ingresso"] / df["PALETS"].replace(0, 1)

# Taxa de ocupação por rota (CODEUT): total de paletes da rota vs capacidade do veículo
rota_totais = (
    df.groupby("CODEUT")
    .agg(total_palets=("PALETS", "sum"), capacidade_rota=("capacidade_norm", "first"))
    .reset_index()
)
rota_totais["taxa_rota"] = rota_totais["total_palets"] / rota_totais["capacidade_rota"].replace(0, 1) * 100

df = df.merge(rota_totais[["CODEUT", "taxa_rota", "total_palets"]], on="CODEUT", how="left")
df["taxa_ocupacao"] = df["PALETS"] / df["total_palets"].replace(0, 1) * df["taxa_rota"]
df = df.drop(columns=["taxa_rota", "total_palets"])

df["margem"] = df["total_ingresso"] - df["COSTEDT"]
df["margem_por_palete"] = df["ingresso_por_palete"] - df["custo_por_palete"]

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,latitude_destino,longitude_origem_c,latitude_origem_c,longitude_destino_c,latitude_destino_c,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,40.853828,-8.890254,39.100256,-8.48262,40.850927,5.460,31.332933,3.030303,25.872933,25.872933
1,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,40.167186,-8.890254,39.100256,-8.50308,40.113495,6.750,13.3,3.030303,6.55,6.55
2,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,40.667549,-8.890254,39.100256,-8.610888,40.646323,7.975,52.701176,6.060606,89.452351,44.726176
3,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,40.351682,-8.890254,39.100256,-8.608674,40.351682,7.330,42.456427,3.030303,35.126427,35.126427
4,100,CEP,SALVESEN LOGISTICA AZAMBUJA 2 / SALVESEN LOGIS...,"TRANSPARENTODISSEIA - Transportes Unipessoal, ...",0000XXX,0000XXX,0.0,0.53,-0.53,0.02,...,37.029997,-8.890254,39.100256,-7.825724,37.039788,0.530,0.0,3.030303,-0.53,-0.53


## 8. Selecionar colunas finais e exportar

In [24]:
colunas_finais = [
    "REFERENCIA", "FENTREGA", "CODEUT", "capacidade_norm", "PALETS", "INGRESODT", "ingresso_danone","total_ingresso", "COSTEDT",
    "PROV_ORIGEN", "LOCORIGEN", "PROV_DESTINO", "LOCDESTINO", "CPOSTAL", "CPOSTAD",
    "longitude_origem", "latitude_origem", "longitude_destino", "latitude_destino",
    "TRANSPORTISTA", "week_day", "week_number", "mes", "mes_nome", "data",
    "dados_validos", "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TRACTORA",
    "PESO_BRUTO", "CODEDT", "CODACT", "TIPOCLIENTE", "TIPOFLUJO",
    "PROV_ENTREGAR", "PAISENTREGAR", "ACTIVIDAD",
    "custo_por_palete", "ingresso_por_palete", "taxa_ocupacao", "margem", "margem_por_palete",
]

df_sel = df[colunas_finais].copy()
df_sel.head()

,REFERENCIA,FENTREGA,CODEUT,capacidade_norm,PALETS,INGRESODT,ingresso_danone,total_ingresso,COSTEDT,PROV_ORIGEN,...,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,5021950798 414190635,2026-02-02,3365589,33.0,1.0,0.0,31.332933,31.332933,5.46,Lisboa,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,5.460,31.332933,3.030303,25.872933,25.872933
1,2021792046,2026-02-02,3365589,33.0,1.0,13.3,<NA>,13.3,6.75,Lisboa,...,NaN,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING,6.750,13.3,3.030303,6.55,6.55
2,5021982094 414203894,2026-02-02,3365589,33.0,2.0,0.0,105.402351,105.402351,15.95,Lisboa,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,7.975,52.701176,6.060606,89.452351,44.726176
3,5021963392 414182550,2026-02-02,3365589,33.0,1.0,0.0,42.456427,42.456427,7.33,Lisboa,...,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL,7.330,42.456427,3.030303,35.126427,35.126427
4,5021973560 414166769,2026-02-02,3365590,33.0,1.0,0.0,<NA>,0.0,0.53,Lisboa,...,TLD PORTUGAL,Directo,Faro,PORTUGAL,DANONE PORTUGAL,0.530,0.0,3.030303,-0.53,-0.53


In [25]:
df_sel.to_parquet(PATHS["parquet"], index=False)

print(f"Ficheiro exportado: {PATHS['parquet']}")
print(f"Linhas: {len(df_sel):,}")
print(f"Colunas: {len(df_sel.columns)}")
print(f"Tamanho: {df_sel.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Ficheiro exportado: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet
Linhas: 101,271
Colunas: 44
Tamanho: 56.96 MB


# Analises


In [26]:
# ==========================
# 1. Marcar entregas Danone
# ==========================
df["tem_danone"] = False
mask_referencia = df["referencia_danone_chave"].notna()

df.loc[mask_referencia, "tem_danone"] = (
    df.loc[mask_referencia]
    .groupby(["referencia_danone_chave", "FENTREGA"])["ingresso_danone"]
    .transform(lambda x: x.notna().any())
)

# ==========================
# 2. Resumo mensal Danone
# ==========================
resumo_danone_mes = (
    df[df["CODACT"].isin([11]) & df["tem_danone"]]
    .groupby("mes")
    .agg(
        ingresso=("ingresso_danone", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)

resumo_danone_mes["margem"] = resumo_danone_mes["ingresso"] - resumo_danone_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_danone_mes[coluna] = resumo_danone_mes[coluna].round(0).map("{:,.0f}".format)

resumo_danone_mes


,mes,ingresso,custo,paletes,peso,margem
0,1,"117,336","117,298","14,534","4,646,811",38
1,2,"107,798","108,663","12,610","3,951,527",-866
2,3,"132,145","134,104","14,739","5,167,799","-1,959"
3,4,"131,149","131,424","14,858","4,661,538",-275
4,5,"125,220","129,713","15,388","4,927,142","-4,494"
5,6,"142,853","139,947","15,179","5,245,161","2,907"
6,7,"154,161","150,398","15,550","5,527,922","3,763"
7,8,"136,893","139,676","16,916","4,734,076","-2,783"
8,9,454,191,75,"2,316",263
9,12,"5,985","5,933",498,"153,524",52


In [27]:
resumo_311_mes = (
    df[df["CODACT"].isin([311])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_311_mes["margem"] = resumo_311_mes["ingresso"] - resumo_311_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_311_mes[coluna] = resumo_311_mes[coluna].round(0).map("{:,.0f}".format)

resumo_311_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"19,652","28,412","3,435","1,079,834","-8,760"
1,2,"18,675","24,506","3,210","832,053","-5,831"
2,3,"20,237","24,005","3,165","955,649","-3,768"
3,4,"23,730","28,626","3,758","1,136,324","-4,896"
4,5,"19,382","23,490","3,208","879,063","-4,108"
5,6,"19,142","32,696","3,371","1,004,582","-13,554"
6,7,"22,879","31,623","3,930","1,262,652","-8,744"
7,8,"21,072","27,269","3,863","1,041,791","-6,197"
8,9,522,995,192,"42,748",-473
9,12,"1,869","2,735",244,"83,981",-866


In [28]:
resumo_311_mes = (
    df[df["CODACT"].isin([74])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_311_mes["margem"] = resumo_311_mes["ingresso"] - resumo_311_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_311_mes[coluna] = resumo_311_mes[coluna].round(0).map("{:,.0f}".format)

resumo_311_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"45,554","49,490","4,969","528,692","-3,937"
1,2,"37,869","46,609","3,930","382,721","-8,740"
2,3,"45,239","49,675","5,837","456,491","-4,437"
3,4,"45,180","50,092","4,603","455,791","-4,912"
4,5,"50,541","56,762","4,180","529,625","-6,220"
5,6,"54,362","55,886","3,999","561,874","-1,524"
6,7,"54,486","58,608","4,373","624,348","-4,122"
7,8,"53,067","53,880","4,246","609,634",-813
8,9,"2,762","2,559",233,"25,499",203
9,12,"3,769","4,433",567,"58,466",-664


In [29]:
resumo_outros_mes = (
    df[~df["CODACT"].isin([11, 311, 13, 74])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_outros_mes["margem"] = resumo_outros_mes["ingresso"] - resumo_outros_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_outros_mes[coluna] = resumo_outros_mes[coluna].round(0).map("{:,.0f}".format)

resumo_outros_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"253,202","201,854","22,348","4,049,911","51,348"
1,2,"231,564","175,640","18,348","3,716,676","55,923"
2,3,"303,499","241,490","23,758","6,490,252","62,009"
3,4,"303,898","252,853","24,344","5,238,258","51,044"
4,5,"297,658","248,036","23,313","5,233,085","49,622"
5,6,"290,807","243,805","22,458","5,286,866","47,002"
6,7,"364,188","272,823","23,860","5,801,374","91,365"
7,8,"319,587","256,129","22,641","6,001,985","63,458"
8,9,"22,128","16,297","2,060","343,587","5,831"
9,12,"18,186","16,051",909,"275,373","2,135"


In [30]:
resumo_total_sem_013 = (
    df[~df["CODACT"].isin([13])]
    .groupby("mes")
    .agg(
        ingresso=("total_ingresso", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_total_sem_013["margem"] = resumo_total_sem_013["ingresso"] - resumo_total_sem_013["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_total_sem_013[coluna] = resumo_total_sem_013[coluna].round(0).map("{:,.0f}".format)

resumo_total_sem_013

,mes,ingresso,custo,paletes,peso,margem
0,1,"435,744","401,222","45,632","10,360,268","34,523"
1,2,"395,905","357,953","38,389","8,928,579","37,953"
2,3,"501,120","452,768","47,830","13,118,146","48,352"
3,4,"503,956","467,728","48,020","11,555,668","36,229"
4,5,"492,801","467,791","46,899","11,624,558","25,010"
5,6,"507,164","486,666","46,101","12,169,237","20,498"
6,7,"595,715","536,096","49,044","13,321,385","59,618"
7,8,"530,619","495,785","49,027","12,540,781","34,834"
8,9,"25,866","28,547","3,649","729,740","-2,682"
9,12,"29,809","29,280","2,222","572,885",529
